In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

c:\Users\E87271\AppData\Local\miniconda3\envs\langchain\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## Indexin

In [4]:
video_id = "6zLCZ_Ic1hI"
try:
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    transcript_text = " ".join(
        snippet.text for snippet in transcript
    )
except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")
except NoTranscriptFound:
    print("No transcript found for this video.")

In [5]:
print(transcript)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='(suspenseful music)', start=10.82, duration=3.0), FetchedTranscriptSnippet(text='- [Nexpo] Saturday.', start=34.92, duration=0.95), FetchedTranscriptSnippet(text='Wildfires are decimating Hawaii.', start=39.45, duration=2.463), FetchedTranscriptSnippet(text='The Barbenheimer phenomenon\nis in full force.', start=42.78, duration=3.9), FetchedTranscriptSnippet(text='And over in an unassuming\ncorner of cyberspace,', start=46.68, duration=2.91), FetchedTranscriptSnippet(text='someone creates a YouTube channel', start=49.59, duration=2.04), FetchedTranscriptSnippet(text='with an urgent call to action.', start=51.63, duration=2.37), FetchedTranscriptSnippet(text='- This matter is of utmost significance', start=54.0, duration=2.37), FetchedTranscriptSnippet(text='for the survival of every\nliving being on this planet.', start=56.37, duration=3.333), FetchedTranscriptSnippet(text="- [Nexpo] It's called the Earth\nSave Science Collabor

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = splitter.split_text(transcript_text)
len(chunks)

44

In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_texts(chunks, embeddings)

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
retriever.invoke('How are the main characters of the story?')

[Document(id='d5ac1dc3-d3b8-4e38-bdc6-c1b0067ec0de', metadata={}, page_content="group. So, who in the world are these people? (suspenseful music) - [Creator 1] Here's a"),
 Document(id='bfb9a93f-d70a-4562-8eea-3c68c46e2457', metadata={}, page_content='the stories quite though- (suspenseful music)'),
 Document(id='162befe0-8ae9-41b9-8ba1-28464315639c', metadata={}, page_content="typical morning routine as someone who owns 11 pets. - [Creator 2] Okay, I've been making this brown sugar mocha dalgona lately and Nescafe is helping me. - November 7th, 155 miles per hour winds just wiped out 80% of the city. - [Creator 3] It is what all people want. We've been conducting a lot\nof surveys for several years. - No day without informing\nabout Creative Society. - [Creator 4] It is not some saintly call. The Creative Society is all of humanity. And when I say all, it means all. (suspenseful music) - [Nexpo] Making its way onto\nevery social media algorithm across the vast ocean of\nthe internet i

In [9]:
llm = ChatOpenAI(model='gpt-4o-mini',temperature=0.2)


In [10]:
prompt = PromptTemplate(
    template = """

        You are a helpful assistant.
        Answer ONLY from the provided context.
        If the context is insuffient, just say you dont know.

        {context}
        Question: {question}
    """, input_variables=['context', 'question']
)

In [11]:
question = 'What are the main story in the video?'
relevant_docs = retriever.invoke(question)

In [12]:
context_text = "\n\n".join(doc.page_content for doc in relevant_docs)
final_prompt = prompt.invoke({'context': context_text, 'question': question})

In [13]:
answer = llm.invoke(final_prompt)

In [14]:
print(answer)

content="The main story in the video revolves around the figure Egon Cholakian and the organization AllatRa, which is portrayed as a potentially fraudulent group that is manipulating public opinion and entangling itself with world governments. The video discusses the bizarre nature of Egon's videos, particularly focusing on his lack of movement in certain clips, and critiques the long, seemingly nonsensical propaganda videos produced by AllatRa. It highlights the impact of such content on audiences and raises concerns about the organization's influence and the individuals behind it, suggesting that shining a light on these groups could help prevent their spread." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 899, 'total_tokens': 1018, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'ca

In [15]:
## chains

In [16]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [17]:
def format_docs(relevant_docs_docs):
    context = "\n\n".join(doc.page_content for doc in relevant_docs)
    return context_text

In [18]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [19]:
parser = StrOutputParser()

In [20]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Who are the main characters and what they do?')